In [1]:
import pandas as pd
import numpy as np

In [2]:
!curl -L -o data.csv 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0   601k      0 --:--:-- --:--:-- --:--:--  600k


In [3]:
df = pd.read_csv("data.csv")

In [4]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
## replacing NAN values with NA for categorical and 0 for numerical

In [33]:
categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

In [9]:
df[categorical] = df[categorical].fillna('NA')

In [10]:
df[numerical] = df[numerical].fillna(0)

In [11]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [12]:
df.shape

(5000, 9)

In [13]:
df['industry'].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

In [14]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [15]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [16]:
corr = df[numerical].corr()

pairs = (
    corr.abs()
        .where(~np.tril(np.ones(corr.shape, dtype=bool)))  # keep upper triangle only
        .stack()
        .sort_values(ascending=False)
)
pairs.head()

interaction_count         lead_score           0.915746
number_of_courses_viewed  lead_score           0.757204
                          interaction_count    0.721609
lead_score                converted            0.483246
interaction_count         converted            0.448624
dtype: float64

In [17]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [18]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [30]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values


In [31]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [21]:
len(df_train), len(df_test), len(df_val)

(3000, 1000, 1000)

In [22]:
from sklearn.metrics import mutual_info_score

In [23]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_train.converted)

In [24]:
mi = df_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [25]:
from sklearn.linear_model import LogisticRegression

In [44]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dict)

In [45]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [46]:
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [47]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [48]:
y_pred = model.predict_proba(X_val)[:,1]

In [49]:
churn_decision = (y_pred>=0.5)

In [50]:
df_val[churn_decision]

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,NA,technology,employed,asia,80457.0,3,5,0.64
2,paid_ads,technology,student,africa,43944.0,2,4,0.49
3,organic_search,finance,employed,europe,73795.0,2,6,0.68
4,paid_ads,finance,NA,north_america,76619.0,1,3,0.40
5,referral,retail,self_employed,south_america,52392.0,5,9,0.93
...,...,...,...,...,...,...,...,...
994,organic_search,finance,employed,south_america,77662.0,4,6,0.73
995,organic_search,retail,employed,europe,46912.0,3,6,0.66
996,paid_ads,technology,student,north_america,0.0,1,5,0.48
998,events,healthcare,student,north_america,40878.0,1,3,0.39


In [56]:
orig_accuracy = (y_val == churn_decision).mean()

In [57]:
# eliminating 

In [62]:
# Model 1 - eliminating 'lead_source' 


categorical = ['industry','employment_status','location']

df_train1 = df_train.drop(columns = 'lead_source')
df_val1 = df_val.drop(columns = 'lead_source')

dv = DictVectorizer(sparse=False)

train_dict = df_train1[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dict)

val_dicts = df_val1[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:,1]

churn_decision = (y_pred>=0.5)

M1_accuracy = (y_val == churn_decision).mean()

orig_accuracy-M1_accuracy

np.float64(0.0030000000000000027)

In [64]:
# Model 2 - eliminating 'number_of_courses_viewed' 


categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','interaction_count','lead_score']
df_train1 = df_train.drop(columns = 'number_of_courses_viewed')
df_val1 = df_val.drop(columns = 'number_of_courses_viewed')

dv = DictVectorizer(sparse=False)

train_dict = df_train1[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dict)

val_dicts = df_val1[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:,1]

churn_decision = (y_pred>=0.5)

M1_accuracy = (y_val == churn_decision).mean()

orig_accuracy-M1_accuracy

np.float64(0.0020000000000000018)

In [65]:
# Model 2 - eliminating 'number_of_courses_viewed' 


categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','lead_score']
df_train1 = df_train.drop(columns = 'interaction_count')
df_val1 = df_val.drop(columns = 'interaction_count')

dv = DictVectorizer(sparse=False)

train_dict = df_train1[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dict)

val_dicts = df_val1[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:,1]

churn_decision = (y_pred>=0.5)

M1_accuracy = (y_val == churn_decision).mean()

orig_accuracy-M1_accuracy

np.float64(0.04400000000000004)

In [66]:
## For Q5, we see that the difference is 0.002 for number_of_courses_viewed feature removal

In [69]:


## Regularized logistic regression
from sklearn.metrics import accuracy_score

categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

results = {}


for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C,  max_iter=1000, random_state=42)
    train_dict = df_train[categorical + numerical].to_dict(orient='records')   
    X_train = dv.fit_transform(train_dict)
    val_dicts = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dicts)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    results[C] = round(acc, 3)
    print(f'C={C:<6} accuracy={acc:.3f}')

    

C=1e-06  accuracy=0.598
C=1e-05  accuracy=0.598
C=0.0001 accuracy=0.613
C=0.001  accuracy=0.645


In [ ]:
## 0.001 gives the highest accuracy of 64.5%